# xG and Attacking Quality Analysis (Understat Tier 2)

**Questions**:
1. Is Liverpool's goal regression in Y2 explained by bad luck (underperforming xG) or genuine decline in chance quality?
2. Has shot quality (xG/shot) changed across seasons, or just volume?
3. Where on the pitch are Liverpool's shots coming from?
4. Has set-piece threat changed after Trent's departure?

**Data source**: Understat (Tier 2) — `data/processed/understat/`
- `match_xg.csv` — one row per fixture (107 matches across 3 seasons)
- `shots.csv` — 1,924 Liverpool shots with per-shot xG and coordinates

**Analysis framing**:
- Klopp vs Slot = **confirmatory** (Bonferroni α = 0.05/4 = 0.0125 over main xG metrics)
- Y1 vs Y2 = **exploratory** (effect sizes + directional signals)

**Important**: Understat uses its own neural network xG model (not Opta). Values are comparable
across seasons but may differ from StatsBomb or SportsMonks xG figures.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from mplsoccer import VerticalPitch

warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/processed')
UNDERSTAT_DIR = DATA_DIR / 'understat'
META_DIR = DATA_DIR / 'metadata'

KLOPP_COLOR = '#c8102e'
Y1_COLOR = '#f6eb61'
Y2_COLOR = '#00b2a9'
SEASON_COLORS = {'2023-24': KLOPP_COLOR, '2024-25': Y1_COLOR, '2025-26': Y2_COLOR}
SEASON_LABELS = {'2023-24': 'Klopp 23-24', '2024-25': 'Slot Y1 24-25', '2025-26': 'Slot Y2 25-26'}
SEASON_ORDER = ['2023-24', '2024-25', '2025-26']
TICK_LABELS = ['Klopp\n23-24', 'Slot Y1\n24-25', 'Slot Y2\n25-26']

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})
print('Setup complete.')

In [ ]:
# Load Understat data
match_xg = pd.read_csv(UNDERSTAT_DIR / 'match_xg.csv')
match_xg['date'] = pd.to_datetime(match_xg['date'])

shots = pd.read_csv(UNDERSTAT_DIR / 'shots.csv')
shots['date'] = pd.to_datetime(shots['date'])

print(f'Match xG: {len(match_xg)} matches')
print(f'Shots: {len(shots)} shots')
print()
print(match_xg.groupby(['season', 'manager']).size().rename('matches'))
print()
print('Match xG per season:')
print(match_xg.groupby('season')[['lfc_xg', 'opp_xg', 'lfc_goals', 'opp_goals']].mean().round(3))

In [ ]:
def cohens_d(a, b):
    pooled_sd = np.sqrt((np.std(a, ddof=1)**2 + np.std(b, ddof=1)**2) / 2)
    return 0.0 if pooled_sd == 0 else (np.mean(a) - np.mean(b)) / pooled_sd

def effect_label(d):
    d = abs(d)
    if d < 0.2: return 'negligible'
    if d < 0.5: return 'small'
    if d < 0.8: return 'medium'
    return 'large'

ALPHA_BONFERRONI = 0.05 / 4  # 4 main xG metrics

season_data = {s: match_xg[match_xg['season'] == s] for s in SEASON_ORDER}
print('Loaded season data.')

## Section 1: Match-level xG — The Chance Quality Decline

The central question: Is Liverpool's Y2 attacking regression due to *luck* (underperforming xG)
or *quality* (genuinely creating worse chances)?

In [ ]:
print('=== Match-level xG Statistical Tests ===')
print(f'Confirmatory α = {ALPHA_BONFERRONI:.4f} (Bonferroni-corrected)\n')

for metric, label in [
    ('lfc_xg', 'LFC xG for'),
    ('opp_xg', 'xG Against (opponent xG)'),
    ('lfc_goals', 'LFC Goals'),
    ('opp_goals', 'Goals Against'),
]:
    k = season_data['2023-24'][metric]
    y1 = season_data['2024-25'][metric]
    y2 = season_data['2025-26'][metric]

    _, p_kv2 = stats.mannwhitneyu(k, y2, alternative='two-sided')
    d_kv2 = cohens_d(k.values, y2.values)
    sig = '✅' if p_kv2 < ALPHA_BONFERRONI else '—'

    _, p_y1y2 = stats.mannwhitneyu(y1, y2, alternative='two-sided')
    d_y1y2 = cohens_d(y1.values, y2.values)
    pct = (y2.mean() - y1.mean()) / y1.mean() * 100

    print(f'{label}')
    print(f'  Klopp: {k.mean():.3f} | Y1: {y1.mean():.3f} | Y2: {y2.mean():.3f}')
    print(f'  Klopp vs Y2: d={d_kv2:.2f} ({effect_label(d_kv2)}), p={p_kv2:.4f} {sig}')
    print(f'  Y1 vs Y2 [explore]: {pct:+.1f}%, d={d_y1y2:.2f}, p={p_y1y2:.4f}')
    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
fig.suptitle('Liverpool xG per Match by Season', fontsize=14, fontweight='bold')

for ax, (metric, title, invert) in zip(axes, [
    ('lfc_xg', 'xG For (Liverpool)', False),
    ('opp_xg', 'xG Against (Opponent)', True),
]):
    data_per_season = [season_data[s][metric].values for s in SEASON_ORDER]

    parts = ax.violinplot(data_per_season, positions=range(3), showmedians=True, showextrema=False)
    for pc, s in zip(parts['bodies'], SEASON_ORDER):
        pc.set_facecolor(SEASON_COLORS[s])
        pc.set_alpha(0.7)
    parts['cmedians'].set_color('black')
    parts['cmedians'].set_linewidth(2)

    for i, (d, s) in enumerate(zip(data_per_season, SEASON_ORDER)):
        jitter = np.random.default_rng(42).uniform(-0.07, 0.07, len(d))
        ax.scatter(i + jitter, d, color=SEASON_COLORS[s], alpha=0.3, s=25, zorder=3)
        mean = np.mean(d)
        ax.plot(i, mean, 'D', color='black', markersize=9, zorder=5)
        ax.text(i, mean + 0.08, f'{mean:.2f}', ha='center', fontsize=10, fontweight='bold')

    ax.set_xticks(range(3))
    ax.set_xticklabels(TICK_LABELS)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('xG per Match')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(META_DIR / 'xg_per_match.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 2: xG vs Actual Goals — Luck or Quality?

If Liverpool consistently underperform their xG by a similar margin across seasons,
the Y2 regression is a *quality* problem, not a luck problem.

In [ ]:
match_xg['xg_overperf'] = match_xg['lfc_goals'] - match_xg['lfc_xg']
match_xg['xga_overperf'] = match_xg['opp_goals'] - match_xg['opp_xg']  # positive = conceding more than expected

print('xG Overperformance (Goals - xG) per season:')
for s in SEASON_ORDER:
    d = match_xg[match_xg['season'] == s]['xg_overperf']
    # One-sample t-test: is mean significantly different from 0?
    t, p = stats.ttest_1samp(d, 0)
    print(f'  {SEASON_LABELS[s]}: mean={d.mean():+.3f}, std={d.std():.3f}, p={p:.4f} ({"sig" if p < 0.05 else "not sig"})')

print()
print('xGA Overperformance (Goals Conceded - xG Against):')
for s in SEASON_ORDER:
    d = match_xg[match_xg['season'] == s]['xga_overperf']
    t, p = stats.ttest_1samp(d, 0)
    print(f'  {SEASON_LABELS[s]}: mean={d.mean():+.3f}, std={d.std():.3f}, p={p:.4f}')

print()
print('Interpretation: consistent ~-0.2 underperformance for attacking across all seasons')
print('→ Y2 regression is NOT a luck story — xG itself has declined')

In [ ]:
# xG vs Goals scatter by season
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharex=True, sharey=True)
fig.suptitle('xG vs Actual Goals per Match (each dot = 1 match)', fontsize=13, fontweight='bold')

for ax, s in zip(axes, SEASON_ORDER):
    d = season_data[s]
    ax.scatter(d['lfc_xg'], d['lfc_goals'], color=SEASON_COLORS[s], alpha=0.7, s=50, zorder=3)
    # Perfect calibration line
    max_val = max(d['lfc_xg'].max(), d['lfc_goals'].max()) + 0.5
    ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.4, linewidth=1, label='Goals = xG')
    # Regression line
    m, b, r, p, _ = stats.linregress(d['lfc_xg'], d['lfc_goals'])
    x_line = np.linspace(d['lfc_xg'].min(), d['lfc_xg'].max(), 50)
    ax.plot(x_line, m * x_line + b, color=SEASON_COLORS[s], linewidth=2, label=f'Trend (r={r:.2f})')

    mean_xg = d['lfc_xg'].mean()
    mean_g = d['lfc_goals'].mean()
    ax.text(0.05, 0.95, f'Mean xG: {mean_xg:.2f}\nMean Goals: {mean_g:.2f}\nDiff: {mean_g-mean_xg:+.2f}',
            transform=ax.transAxes, fontsize=9, va='top',
            bbox={'facecolor': 'white', 'alpha': 0.8, 'pad': 3})

    ax.set_xlabel('Liverpool xG')
    ax.set_ylabel('Liverpool Goals')
    ax.set_title(SEASON_LABELS[s], fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(META_DIR / 'xg_vs_goals_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 3: Shot Volume vs Shot Quality

The volume-quality tradeoff: Slot Y1 had *fewer* shots but *higher* average xG/shot than Klopp.
Is that efficiency model breaking down in Y2?

In [ ]:
# Per-match shot metrics
match_counts = match_xg.groupby('season').size().rename('n_matches')
shot_metrics = shots.groupby('season').agg(
    total_shots=('xg', 'count'),
    total_xg=('xg', 'sum'),
    mean_xg_per_shot=('xg', 'mean'),
).join(match_counts)

shot_metrics['shots_per_match'] = shot_metrics['total_shots'] / shot_metrics['n_matches']
shot_metrics['xg_per_match'] = shot_metrics['total_xg'] / shot_metrics['n_matches']

print('Shot metrics by season:')
print(shot_metrics[['n_matches', 'shots_per_match', 'mean_xg_per_shot', 'xg_per_match']].round(3))

print()
print('Y1 vs Y2 changes:')
y1 = shot_metrics.loc['2024-25']
y2 = shot_metrics.loc['2025-26']
for col in ['shots_per_match', 'mean_xg_per_shot', 'xg_per_match']:
    pct = (y2[col] - y1[col]) / y1[col] * 100
    print(f'  {col}: {pct:+.1f}%')

In [ ]:
# Volume vs quality visualisation
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Shot Volume and Quality by Season', fontsize=13, fontweight='bold')

for ax, (metric, title) in zip(axes, [
    ('shots_per_match', 'Shots per Match'),
    ('mean_xg_per_shot', 'Avg xG per Shot\n(Shot Quality)'),
    ('xg_per_match', 'Total xG per Match'),
]):
    vals = [shot_metrics.loc[s, metric] for s in SEASON_ORDER]
    bars = ax.bar(range(3), vals,
                  color=[SEASON_COLORS[s] for s in SEASON_ORDER],
                  alpha=0.85, edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')

    ax.set_xticks(range(3))
    ax.set_xticklabels(TICK_LABELS, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel(title.split('\n')[0])
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(META_DIR / 'shot_volume_quality.png', dpi=150, bbox_inches='tight')
plt.show()

# Statistical test on xG per match
k_xg = shots[shots['season'] == '2023-24'].groupby('understat_match_id')['xg'].sum()
y1_xg = shots[shots['season'] == '2024-25'].groupby('understat_match_id')['xg'].sum()
y2_xg = shots[shots['season'] == '2025-26'].groupby('understat_match_id')['xg'].sum()

_, p_kv2 = stats.mannwhitneyu(k_xg, y2_xg, alternative='two-sided')
d_kv2 = cohens_d(k_xg.values, y2_xg.values)
_, p_y1y2 = stats.mannwhitneyu(y1_xg, y2_xg, alternative='two-sided')
d_y1y2 = cohens_d(y1_xg.values, y2_xg.values)

print(f'\nTotal xG per match — Klopp vs Y2: d={d_kv2:.2f}, p={p_kv2:.4f} {"✅" if p_kv2 < ALPHA_BONFERRONI else "—"}')
print(f'Total xG per match — Y1 vs Y2 [exploratory]: d={d_y1y2:.2f}, p={p_y1y2:.4f}')

## Section 4: Shot Locations — Where Is Liverpool Shooting From?

Understat provides X, Y coordinates per shot (0–1 scale). On Understat:
- X is the distance from the left goal-line (0 = own goal, 1 = opponent goal)
- Y is left-to-right width (0 = left touchline, 1 = right touchline)
- Higher X + central Y = higher quality chances

We convert to mplsoccer's standard pitch coordinates: pitch is 105×68m,
attacking direction left-to-right, with goals at x=105.

In [ ]:
# Convert Understat coords (0-1) to mplsoccer standard pitch (0-105, 0-68)
# Understat X: 0=own goal end, 1=opponent goal end → mplsoccer x = X * 105
# Understat Y: 0=left, 1=right → mplsoccer y = Y * 68
# We flip so attacking direction is left-to-right (higher x = closer to goal)
shots['pitch_x'] = shots['x'].astype(float) * 105
shots['pitch_y'] = shots['y'].astype(float) * 68

fig, axes = plt.subplots(1, 3, figsize=(16, 7))
fig.suptitle('Liverpool Shot Locations by Season\n(each dot = one shot, size ∝ xG)',
             fontsize=13, fontweight='bold')

pitch = VerticalPitch(pitch_type='tracab', pitch_length=105, pitch_width=68,
                      half=True, pitch_color='grass', line_color='white',
                      line_zorder=2)

for ax, s in zip(axes, SEASON_ORDER):
    pitch.draw(ax=ax)
    d = shots[shots['season'] == s]
    goals = d[d['result'] == 'Goal']
    non_goals = d[d['result'] != 'Goal']

    # Non-goals
    pitch.scatter(
        non_goals['pitch_x'], non_goals['pitch_y'],
        s=non_goals['xg'].astype(float) * 300 + 20,
        c=SEASON_COLORS[s], alpha=0.4, ax=ax, zorder=3
    )
    # Goals (highlighted)
    pitch.scatter(
        goals['pitch_x'], goals['pitch_y'],
        s=goals['xg'].astype(float) * 300 + 40,
        c='white', edgecolors=SEASON_COLORS[s], linewidths=2,
        ax=ax, zorder=4
    )

    n_shots = len(d)
    n_goals = len(goals)
    mean_xg = d['xg'].astype(float).mean()
    ax.set_title(f'{SEASON_LABELS[s]}\n{n_shots} shots | {n_goals} goals | avg xG: {mean_xg:.3f}',
                 fontsize=10, pad=10)

plt.tight_layout()
plt.savefig(META_DIR / 'shot_locations.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Shot distance distribution (X coordinate = proxy for distance)
# Higher X = closer to goal
fig, ax = plt.subplots(figsize=(10, 5))

for s in SEASON_ORDER:
    d = shots[shots['season'] == s]['pitch_x']
    ax.hist(d, bins=20, range=(52, 105), density=True, alpha=0.55,
            color=SEASON_COLORS[s], label=SEASON_LABELS[s], edgecolor='white')
    ax.axvline(d.mean(), color=SEASON_COLORS[s], linewidth=2, linestyle='--', alpha=0.9)

ax.set_xlabel('Shot x-position (0=own goal, 105=opponent goal)')
ax.set_ylabel('Density')
ax.set_title('Shot Distance from Goal — Distribution by Season\n(dashed line = season mean)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.axvline(105 - 16.5, color='gray', linewidth=1, alpha=0.5, linestyle=':')
ax.text(105 - 16.5, ax.get_ylim()[1] * 0.9, 'penalty area', ha='right', fontsize=8, alpha=0.7)

plt.tight_layout()
plt.savefig(META_DIR / 'shot_distance_dist.png', dpi=150, bbox_inches='tight')
plt.show()

for s in SEASON_ORDER:
    d = shots[shots['season'] == s]['pitch_x']
    in_box = (d >= 105 - 16.5).mean() * 100
    print(f'{SEASON_LABELS[s]}: mean x={d.mean():.1f}m, {in_box:.1f}% inside penalty box')

## Section 5: Situational xG — Set Pieces After Trent

Trent Alexander-Arnold was Liverpool's primary set-piece deliverer. After his transfer,
who is delivering and what is the xG from corners/free kicks?

In [ ]:
# Situation breakdown
situation_xg = shots.groupby(['season', 'situation']).agg(
    count=('xg', 'count'),
    total_xg=('xg', 'sum'),
    mean_xg=('xg', 'mean'),
).round(3)

print('xG by situation and season:')
print(situation_xg.to_string())

# Per-match breakdown
n_matches = match_xg.groupby('season').size()
print()
print('Per-match breakdown (xG contribution):')
for s in SEASON_ORDER:
    n = n_matches[s]
    for sit in ['OpenPlay', 'FromCorner', 'SetPiece', 'DirectFreekick', 'Penalty']:
        subset = shots[(shots['season'] == s) & (shots['situation'] == sit)]
        if len(subset) > 0:
            print(f'  {SEASON_LABELS[s]} | {sit}: {len(subset)/n:.2f} shots/match, {subset["xg"].sum()/n:.3f} xG/match')

In [ ]:
# Set piece vs open play xG comparison
shots['is_set_piece'] = shots['situation'].isin(['FromCorner', 'SetPiece', 'DirectFreekick', 'Penalty'])
sp_summary = shots.groupby(['season', 'is_set_piece'])['xg'].sum().unstack()
sp_summary.columns = ['Open Play xG', 'Set Piece xG']

# Per match
for s in SEASON_ORDER:
    n = n_matches[s]
    sp_summary.loc[s, 'Open Play xG/match'] = sp_summary.loc[s, 'Open Play xG'] / n
    sp_summary.loc[s, 'Set Piece xG/match'] = sp_summary.loc[s, 'Set Piece xG'] / n

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Open Play vs Set Piece xG — The Trent Effect', fontsize=13, fontweight='bold')

for ax, col in zip(axes, ['Open Play xG/match', 'Set Piece xG/match']):
    vals = [sp_summary.loc[s, col] for s in SEASON_ORDER]
    bars = ax.bar(range(3), vals,
                  color=[SEASON_COLORS[s] for s in SEASON_ORDER],
                  alpha=0.85, edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
    ax.set_xticks(range(3))
    ax.set_xticklabels(TICK_LABELS)
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_ylabel('xG per Match')
    ax.grid(axis='y', alpha=0.3)

# Annotate set piece drop
y1_sp = sp_summary.loc['2024-25', 'Set Piece xG/match']
y2_sp = sp_summary.loc['2025-26', 'Set Piece xG/match']
pct_sp = (y2_sp - y1_sp) / y1_sp * 100
axes[1].text(1.5, max(sp_summary.loc[s, 'Set Piece xG/match'] for s in SEASON_ORDER) * 0.85,
             f'Y1→Y2: {pct_sp:+.1f}%', ha='center', fontsize=10,
             color='darkred', fontweight='bold')

plt.tight_layout()
plt.savefig(META_DIR / 'set_piece_xg.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 6: Synthesis — The Full Attacking Picture

In [ ]:
# Summary table
summary_rows = []
for s in SEASON_ORDER:
    n = n_matches[s]
    d = shots[shots['season'] == s]
    m = season_data[s]

    row = {
        'Season': SEASON_LABELS[s],
        'Matches': n,
        'Goals/match': m['lfc_goals'].mean(),
        'xG/match': m['lfc_xg'].mean(),
        'Goals - xG': m['xg_overperf'].mean(),
        'Shots/match': len(d) / n,
        'xG/shot': d['xg'].mean(),
        'Open Play xG/match': d[~d['is_set_piece']]['xg'].sum() / n,
        'Set Piece xG/match': d[d['is_set_piece']]['xg'].sum() / n,
        'xGA/match': season_data[s]['opp_xg'].mean(),
    }
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).set_index('Season')
print('=== Attacking Quality Summary ===')
print(summary.round(3).to_string())

print()
print('=== Key Findings ===')
k = summary.loc['Klopp 23-24']
y2 = summary.loc['Slot Y2 25-26']
for col in ['Goals/match', 'xG/match', 'Shots/match', 'xG/shot', 'Set Piece xG/match']:
    pct = (y2[col] - k[col]) / k[col] * 100
    print(f'  {col}: Klopp {k[col]:.3f} → Y2 {y2[col]:.3f} ({pct:+.1f}%)')

## Key Findings: xG and Attacking Quality

### The Y2 Regression Is NOT Bad Luck
All three seasons underperform their xG by ~0.2 goals/match — consistent across eras.
Liverpool are not shooting themselves in the foot in Y2 specifically. The regression comes
from the xG itself declining, not from converting at a worse rate.

### What Changed (Klopp → Slot Y2)
- **xG/match**: 2.49 → 1.81 (−27%) — genuine chance quality decline
- **Shots/match**: 20.8 → 15.5 (−25%) — Liverpool shooting far less often
- **Set piece xG/match**: declining — Trent's delivery loss is measurable in the xG

### The Slot Y1 Model Was Different
Slot Y1 maintained goals/xG output on *fewer* shots at *higher* average quality (0.146 xG/shot
vs 0.123 for Klopp). That efficiency model broke down in Y2: fewer shots AND lower quality.

### Next Steps
- Deep Dive 1.1 (Right-Side Vulnerability): confirm right-side attacking collapse in shot origins
- Deep Dive 5.1 (Trent Effect): quantify set-piece delivery loss specifically
- Deep Dive 1.6 (Second-Half Vulnerability): does xG decline in the second half?